In [1]:
# =========================================================
# COSINE SIMILARITY
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity

from src.preprocessing.clean_text import clean_text
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

from supabase import create_client, Client
import uuid
from datetime import datetime



In [2]:
# =========================================================
# KONEKSI SUPABASE
# =========================================================
SUPABASE_URL = 'https://bnuzmrtiaciqlotxcgot.supabase.co'
SUPABASE_KEY = 'sb_publishable_Z8M8GISPVKMp1SGrQHrlLg_AZa8EOo-'
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Koneksi Supabase berhasil!')


✅ Koneksi Supabase berhasil!


In [3]:
# =========================================================
# LOAD DATA
# =========================================================
base_dir = os.path.abspath('..')
tfidf_dir = os.path.join(base_dir, 'data', 'tfidf')

# Load vectorizer
with open(os.path.join(tfidf_dir, 'vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)

# Load matriks TF-IDF
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, 'tfidf_matrix.npz'))

# Load data artikel lengkap
doc_index = pd.read_csv(os.path.join(base_dir, 'data', 'cleaned_papers.csv'))

stop_words = get_stopwords()

print(f'✅ Matriks TF-IDF: {tfidf_matrix.shape}')
print(f'✅ Dokumen: {len(doc_index)} artikel')


✅ Matriks TF-IDF: (200, 1718)
✅ Dokumen: 200 artikel


In [4]:
# =========================================================
# PREPROCESSING QUERY + TOKEN HELPER
# =========================================================
def preprocess_query(query: str) -> str:
    text = clean_text(query)
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    if all(token.isascii() for token in tokens):
        return ' '.join(tokens)
    return ' '.join(stemming(tokens))

def tokenize_text_for_tf(text: str) -> list:
    text = clean_text(str(text))
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    if all(token.isascii() for token in tokens):
        return tokens
    return stemming(tokens)

def count_term_frequency_consistent(text: str, query_terms: list) -> int:
    tokens = tokenize_text_for_tf(text)
    return sum(tokens.count(term) for term in query_terms)

def is_direct_pdf(url: str) -> bool:
    if pd.isna(url):
        return False
    u = str(url).lower()
    return (
        u.endswith(".pdf")
        or "/pdf/" in u
        or "arxiv.org/pdf" in u
        or "pmc.ncbi.nlm.nih.gov" in u
        or "jmlr.org" in u
    )

In [5]:
# =========================================================
# FUNGSI SEARCH
# =========================================================
def search(query: str, top_k: int = 10) -> pd.DataFrame:
    processed_query = preprocess_query(query).strip()
    if not processed_query:
        return pd.DataFrame(columns=[
            'id','title','abstract','authors','year','source','category',
            'pdf_url','url','scrape_status','similarity_score','term_frequency',
            'occurrence','is_pdf','access_url','rank'
        ])

    query_terms = processed_query.split()
    query_vector = vectorizer.transform([processed_query])

    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    ranked_idx = np.argsort(scores)[::-1]

    results = doc_index.iloc[ranked_idx][[
        'id','title','abstract','authors','year',
        'source','category','pdf_url','url','scrape_status'
    ]].copy()

    results['similarity_score'] = scores[ranked_idx]
    results = results[results['similarity_score'] > 0].copy()
    results = results.head(top_k).reset_index(drop=True)

    results['abstract'] = results['abstract'].fillna("")
    results['term_frequency'] = results['abstract'].apply(
        lambda x: count_term_frequency_consistent(x, query_terms)
    )
    results['occurrence'] = results['term_frequency']

    results['is_pdf'] = results['pdf_url'].apply(is_direct_pdf)
    results['access_url'] = results.apply(
        lambda r: r['pdf_url'] if r['is_pdf'] else r['url'],
        axis=1
    )
    results['rank'] = range(1, len(results) + 1)
    return results

print('✅ Fungsi search siap!')

✅ Fungsi search siap!


In [6]:
# =========================================================
# SAVE TO SUPABASE (SIMILARITY_RESULTS ONLY)
# =========================================================
def save_results_to_supabase(results: pd.DataFrame, query: str):
    if results.empty:
        print('⚠️ Tidak ada data untuk disimpan ke similarity_results')
        return

    records = []
    for _, row in results.iterrows():
        records.append({
            'id': str(uuid.uuid4()),
            'article_id': int(row['id']),
            'compared_text': query,
            'similarity_score': float(round(row['similarity_score'], 6)),
            'created_at': datetime.utcnow().isoformat()
        })

    try:
        supabase.table('similarity_results').insert(records).execute()
        print(f'✅ {len(records)} data tersimpan ke similarity_results (query: "{query}")')
    except Exception as e:
        print(f'❌ Gagal simpan ke Supabase: {e}')

In [7]:
# =========================================================
# RUN SEARCH + SAVE CSV + SAVE SUPABASE
# =========================================================
queries_proposal = [
    "machine learning",
    "web development",
    "cyber security",
    "mobile application"
]

results_dir = os.path.join(base_dir, 'data', 'cosine_results')
os.makedirs(results_dir, exist_ok=True)

for query in queries_proposal:
    print(f'\n{"="*60}')
    print(f'🔍 Query: "{query}"')

    results = search(query, top_k=10)

    filename = f'similarity_score_{query.replace(" ", "_")}.csv'
    save_path = os.path.join(results_dir, filename)
    results.to_csv(save_path, index=False)

    if results.empty:
        print(f'⚠️ Tidak ada hasil. File kosong dibuat: {save_path}')
        continue

    total_occ = int(results['term_frequency'].sum())
    paper_count = int(len(results))

    print(f'{"Rank":<5} {"Score":>8} {"TF":>5}  Judul')
    print('-'*80)
    for _, row in results.iterrows():
        print(f'{int(row["rank"]):<5} {row["similarity_score"]:>8.4f} {int(row["term_frequency"]):>5}  {str(row["title"])[:55]}')

    print(f'\n✅ CSV tersimpan : {save_path}')
    print(f'📊 Paper count   : {paper_count}')
    print(f'📊 Total TF      : {total_occ}')

    save_results_to_supabase(results, query)

print('\n🎉 Selesai!')


🔍 Query: "machine learning"
Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.6232     9  Machine learning and deep learning: A review of methods
2       0.6099     9  When machine learning meets privacy: A survey and outlo
3       0.5994     9  Machine learning and deep learning: C. Janiesch et al.
4       0.5984     8  An overview of machine learning classification techniqu
5       0.5634     6  Scientific machine learning benchmarks
6       0.5581     7  Financial applications of machine learning: A literatur
7       0.5327     6  Machine learning in chemical engineering: A perspective
8       0.5275     6  Machine learning foundations
9       0.5235     6  Machine learning and applications in microbiology
10      0.5065     6  Using machine learning to detect misstatements

✅ CSV tersimpan : d:\Tugas Akhir\paperci_artikel\backend\data\cosine_results\similarity_score_machine_learning.csv
📊 Paper count   : 10
📊 Tot

C:\Users\hp_\AppData\Local\Temp\ipykernel_15124\2823116308.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()


✅ 10 data tersimpan ke similarity_results (query: "machine learning")

🔍 Query: "web development"
Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.5162     4  Accessible web development: Opportunities to improve th
2       0.5025     5  AI AND WEB DEVELOPMENT
3       0.4867     6  The rise of disappearing frameworks in web development
4       0.4540     5  Python for web development
5       0.4233     4  Machine learning for web development: A fusion
6       0.4141     4  Current web development technologies: a comparative rev
7       0.4122     5  Llms in web development: Evaluating llm-generated php c
8       0.4093     4  Cognitive disabilities and web accessibility: a survey 
9       0.4060     5  Web Developer and Tech Preneurs
10      0.3991     5  Program Pelatihan Web Development untuk Komunitas Maya

✅ CSV tersimpan : d:\Tugas Akhir\paperci_artikel\backend\data\cosine_results\similarity_score_web_development

C:\Users\hp_\AppData\Local\Temp\ipykernel_15124\2823116308.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()
C:\Users\hp_\AppData\Local\Temp\ipykernel_15124\2823116308.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()


✅ 10 data tersimpan ke similarity_results (query: "cyber security")

🔍 Query: "mobile application"
Rank     Score    TF  Judul
--------------------------------------------------------------------------------
1       0.4752     6  Deep learning methods for accurate skin cancer recognit
2       0.4291     6  User interface design & evaluation of mobile applicatio
3       0.4269     4  User experience analysis on mobile application design u
4       0.4231     5  Consumer adoption of the Uber mobile application: Insig
5       0.4033     4  A new mobile application of agricultural pests recognit
6       0.3980     5  Design and implementation of smart hydroponics farming 
7       0.3795     4  Towards a new learning experience through a mobile appl
8       0.3487     4  Human monkeypox classification from skin lesion images 
9       0.3345     4  The role of mobile application acceptance in shaping e-
10      0.3163     4  Enhancing English vocabulary learning through mobile ap

✅ CSV tersi

C:\Users\hp_\AppData\Local\Temp\ipykernel_15124\2823116308.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat()
